# Retrieve metadata for a validation process

This tutorial shows how pysdmx can be used to retrieve the structures required to validate data.

## Step 1: Connect to the service that contains the required structures

The client can connect to any compatible SDMX-RESTv2 service.

In [9]:
from pathlib import Path

from pysdmx.api.fmr import RegistryClient
from pysdmx.io import read_sdmx

client = RegistryClient("https://registry.sdmx.org/sdmx/v2")

## Step 2: Retrieve the required schema

In [10]:
schema = client.get_schema("dataflow", "ESTAT", "BPM6_BOP_M", "2.6.0")
print(schema)

context: dataflow, agency: ESTAT, id: BPM6_BOP_M, components: 34 components, version: 2.6.0, artefacts: 28 artefacts, generated: 2026-11-19 08:51:06.251898+00:00


## Step 3: Inspect the schema

Notice the list of components, their data type, enumerations, and facets.

In [11]:
print([component.id for component in schema.components])

frequency = schema.components["FREQ"]
print("FREQ dimension:")
print(f"  - Data type: {frequency.dtype}")
print(f"  - Enumeration: {[code.id for code in frequency.enumeration]}")
print(f"  - Facets: {frequency.facets}")

['FREQ', 'ADJUSTMENT', 'REF_AREA', 'COUNTERPART_AREA', 'REF_SECTOR', 'COUNTERPART_SECTOR', 'FLOW_STOCK_ENTRY', 'ACCOUNTING_ENTRY', 'INT_ACC_ITEM', 'FUNCTIONAL_CAT', 'INSTR_ASSET', 'MATURITY', 'UNIT_MEASURE', 'CURRENCY_DENOM', 'VALUATION', 'COMP_METHOD', 'TYPE_ENTITY', 'TIME_PERIOD', 'OBS_VALUE', 'TIME_FORMAT', 'OBS_STATUS', 'CONF_STATUS', 'PRE_BREAK_VALUE', 'COMMENT_OBS', 'COMMENT_TS', 'TITLE', 'UNIT_MULT', 'DECIMALS', 'TIME_PER_COLLECT', 'REF_PERIOD_DETAIL', 'COMPILING_ORG', 'DATA_COMP', 'DISS_ORG', 'TITLE_COMPL']
FREQ dimension:
  - Data type: String
  - Enumeration: ['M']
  - Facets: min_length: 1, max_length: 1


Notice the list of structures used to generate the schema (`artefacts`).

In [12]:
for artefact in schema.artefacts:
    print(artefact)

urn:sdmx:org.sdmx.infomodel.datastructure.Dataflow=ESTAT:BPM6_BOP_M(2.6.0)
urn:sdmx:org.sdmx.infomodel.datastructure.DataStructure=IMF:BOP(2.6.0)
urn:sdmx:org.sdmx.infomodel.registry.DataConstraint=ESTAT:CR_BPM6_BOP_M(2.6.0)
urn:sdmx:org.sdmx.infomodel.base.AgencyScheme=SDMX:AGENCIES(1.0)
urn:sdmx:org.sdmx.infomodel.codelist.Codelist=ESTAT:CL_INSTR_ASSET(1.15.0)
urn:sdmx:org.sdmx.infomodel.codelist.Codelist=ESTAT:CL_MATURITY(1.8)
urn:sdmx:org.sdmx.infomodel.codelist.Codelist=ESTAT:CL_OBS_STATUS(2.3)
urn:sdmx:org.sdmx.infomodel.codelist.Codelist=ESTAT:CL_REF_PERIOD_DTL(1.0)
urn:sdmx:org.sdmx.infomodel.codelist.Codelist=ESTAT:CL_SECTOR(1.14)
urn:sdmx:org.sdmx.infomodel.codelist.Codelist=ESTAT:CL_VALUATION(1.6)
urn:sdmx:org.sdmx.infomodel.codelist.Codelist=IMF:CL_ACCOUNTS_ITEM(1.8.0)
urn:sdmx:org.sdmx.infomodel.codelist.Codelist=IMF:CL_ACCOUNT_ENTRY(1.5.0)
urn:sdmx:org.sdmx.infomodel.codelist.Codelist=IMF:CL_ADJUSTMENT(1.4)
urn:sdmx:org.sdmx.infomodel.codelist.Codelist=IMF:CL_AREA(2.0.0)


Notice the attribute relationships.

In [13]:
for attribute, relationship in schema.attribute_relationships.items():
    print(f"{attribute}: {relationship.name}")

TIME_FORMAT: DIMENSIONS
OBS_STATUS: OBSERVATION
CONF_STATUS: OBSERVATION
PRE_BREAK_VALUE: OBSERVATION
COMMENT_OBS: OBSERVATION
COMMENT_TS: DIMENSIONS
TITLE: DIMENSIONS
UNIT_MULT: DIMENSIONS
DECIMALS: DIMENSIONS
TIME_PER_COLLECT: DIMENSIONS
REF_PERIOD_DETAIL: DIMENSIONS
COMPILING_ORG: DIMENSIONS
DATA_COMP: DIMENSIONS
DISS_ORG: DIMENSIONS
TITLE_COMPL: DIMENSIONS


## Step 4: Load a data file and inspect its data

Note: The CSV file was generated automatically, i.e. there is no guarantee the data are meaningful.

In [14]:
file_path = Path("../samples/bpm6_bop_m.csv")
if not file_path.exists():
    file_path = Path("docs/tutorials/samples/bpm6_bop_m.csv")
submission = read_sdmx(file_path)
display(submission.data[0])

PandasDataset(structure='Dataflow=ESTAT:BPM6_BOP_M(2.6.0)', action=ActionType.Replace, data=  FREQ ADJUSTMENT REF_AREA COUNTERPART_AREA REF_SECTOR COUNTERPART_SECTOR  \
0    M          N       HU               1A         S1                S11   
1    M          N       GR               1A         S1                S11   
2    M          N       R1               1A         S1                S11   
3    Q          N       R1               1A         S1                S11   

  FLOW_STOCK_ENTRY ACCOUNTING_ENTRY INT_ACC_ITEM FUNCTIONAL_CAT  ... TITLE  \
0               K7                A           CA              D  ...         
1               K7                A           CA              D  ...         
2               K7                A           CA              D  ...         
3               K7                A           CA              D  ...         

  UNIT_MULT DECIMALS TIME_PER_COLLECT REF_PERIOD_DETAIL COMPILING_ORG  \
0         6        0                S                     

As you can see, the data are represented as Pandas DataFrames.

## Write your own code to validate a data message using the schema

Note: pysdmx will soon offer a `validate()` function!

In [15]:
for ds in submission.data:
    df = ds.data

    # For example, check that values belong to the relevant enumerations.
    for column in df.columns:
        component = schema.components[column]
        if component and component.enumeration:
            allowed_values = {code.id for code in component.enumeration}
            values = {value for value in df[column].dropna() if value != ""}
            unexpected = values - allowed_values
            if unexpected:
                print(f"Found invalid values for {component.id}: {unexpected}")

Found invalid values for FREQ: {'Q'}
